# FitFuel — Wan2.1 T2V 1.3B on Google Colab\n\nThis notebook connects the Colab experiment to the public `fitness-shorts-bot` GitHub repository. It keeps model weights out of GitHub, records RAM usage, and optionally sends the run log back to the separate `colab-runs` branch.\n\n**Use a GPU runtime.** Runtime → Change runtime type → T4 GPU. The notebook does not change or run the production YouTube workflow.

In [ ]:
import os, subprocess, pathlib, sys, json, time, datetime, shutil\nREPO_URL = 'https://github.com/6d28nf7wz9-web/fitness-shorts-bot.git'\nREPO_BRANCH = 'feat/colab-wan21-bridge'\nREPO_DIR = '/content/fitness-shorts-bot'\nWAN_DIR = '/content/Wan2.1'\nCKPT_DIR = '/content/Wan2.1-T2V-1.3B'\n\nif os.path.isdir(os.path.join(REPO_DIR, '.git')):\n    subprocess.run(['git','-C',REPO_DIR,'fetch','origin',REPO_BRANCH], check=True)\n    subprocess.run(['git','-C',REPO_DIR,'checkout',REPO_BRANCH], check=True)\n    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only','origin',REPO_BRANCH], check=True)\nelse:\n    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,REPO_DIR], check=True)\n\nif not os.path.isdir(os.path.join(WAN_DIR, '.git')):\n    subprocess.run(['git','clone','https://github.com/Wan-Video/Wan2.1.git',WAN_DIR], check=True)\nelse:\n    print('Reusing existing official Wan2.1 checkout:', WAN_DIR)\n\nprint('Bridge branch:', REPO_BRANCH)\nprint('Project repo:', REPO_DIR)\nprint('Wan source:', WAN_DIR)\nprint('Checkpoint folder exists:', os.path.isdir(CKPT_DIR))

## 2. Runtime and checkpoint preflight\n\nThis cell does not download or delete model files. If the checkpoint folder is missing after a Colab reset, restore it to `/content/Wan2.1-T2V-1.3B` before generation.

In [ ]:
import torch, psutil, shutil, os\nvm = psutil.virtual_memory()\nprint('Python:', sys.version.split()[0])\nprint('PyTorch:', torch.__version__)\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))\n    print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))\nprint('RAM total / available (GB):', round(vm.total/1024**3,2), '/', round(vm.available/1024**3,2))\nprint('Disk free (GB):', round(shutil.disk_usage('/content').free/1024**3,2))\nprint('Wan source exists:', os.path.isfile(os.path.join(WAN_DIR,'generate.py')))\nprint('Checkpoint folder exists:', os.path.isdir(CKPT_DIR))\nif os.path.isdir(CKPT_DIR):\n    for name in ['models_t5_umt5-xxl-enc-bf16.pth','Wan2.1_VAE.pth','diffusion_pytorch_model.safetensors']:\n        p=os.path.join(CKPT_DIR,name)\n        print(name, 'FOUND' if os.path.isfile(p) else 'MISSING', (round(os.path.getsize(p)/1024**3,2) if os.path.isfile(p) else ''))\nassert torch.cuda.is_available(), 'Select a GPU runtime in Colab first.'\nassert os.path.isfile(os.path.join(WAN_DIR,'generate.py')), 'Wan2.1 source is missing.'\nassert os.path.isdir(CKPT_DIR), 'Checkpoint folder missing. Restore/download it before running generation.'

## 3. Apply the low-copy T5 checkpoint load patch\n\nThe patch uses PyTorch memory-mapped checkpoint loading and `assign=True`. It is idempotent. This reduces an extra full checkpoint copy; it does **not** guarantee the 12 GB Colab RAM limit is sufficient. We will measure actual RAM during the run.

In [ ]:
t5_file = os.path.join(WAN_DIR, 'wan/modules/t5.py')\nwith open(t5_file, 'r', encoding='utf-8') as f: source = f.read()\nold = \"model.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))\"\nnew = \"model.load_state_dict(\\n            torch.load(checkpoint_path, map_location='cpu', mmap=True),\\n            assign=True\\n        )\"\nif new in source:\n    print('T5 mmap + assign patch already present.')\nelif old in source:\n    shutil.copy2(t5_file, t5_file + '.pre_colab_bridge_backup')\n    source = source.replace(old, new, 1)\n    with open(t5_file, 'w', encoding='utf-8') as f: f.write(source)\n    print('Applied T5 mmap + assign patch; backup saved beside t5.py.')\nelse:\n    raise RuntimeError('T5 loader differs from expected source. No automatic edit was made; inspect wan/modules/t5.py first.')\nprint('T5 patch check complete.')

## 4. Generate a short FitFuel test and monitor RAM\n\nThe test is intentionally small (17 frames, 20 sampling steps). The output and full console log are saved under `/content`. Keep the Colab tab open until the cell completes.

In [ ]:
import subprocess, threading, psutil, time, os, datetime\nos.chdir(WAN_DIR)\nOUTPUT = '/content/fitfuel_scene_001.mp4'\nLOG = '/content/fitfuel_wan21_run.log'\ncmd = [sys.executable, 'generate.py', '--task','t2v-1.3B', '--size','832*480', '--frame_num','17', '--ckpt_dir',CKPT_DIR, '--offload_model','True', '--t5_cpu', '--sample_shift','8', '--sample_guide_scale','6', '--sample_steps','20', '--save_file',OUTPUT, '--prompt','A cinematic 3D anatomical fitness explainer showing a muscular human figure performing a dumbbell bicep curl. Realistic muscle fibers, detailed anatomy, dark professional studio background, smooth camera movement.']\nprint('Starting:', ' '.join(cmd))\nproc = subprocess.Popen(cmd, cwd=WAN_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\nstop_monitor = threading.Event()\ndef monitor_ram():\n    try: child = psutil.Process(proc.pid)\n    except psutil.Error: child = None\n    while not stop_monitor.wait(5):\n        vm = psutil.virtual_memory()\n        try: rss = child.memory_info().rss/1024**3 if child else 0\n        except psutil.Error: rss = 0\n        print(f'[RAM] available={vm.available/1024**3:.2f} GB | process RSS={rss:.2f} GB')\nthread = threading.Thread(target=monitor_ram, daemon=True); thread.start()\nwith open(LOG, 'w', encoding='utf-8') as log:\n    for line in proc.stdout:\n        print(line, end='')\n        log.write(line); log.flush()\n    return_code = proc.wait()\nstop_monitor.set(); thread.join(timeout=2)\nprint('Process exit code:', return_code)\nprint('Output exists:', os.path.isfile(OUTPUT))\nif os.path.isfile(OUTPUT): print('Output size (MB):', round(os.path.getsize(OUTPUT)/1024**2,2))\nif return_code != 0: print('Generation failed. Inspect the saved log:', LOG)\nelse: print('Generation completed. Verify the preview in the next cell.')

In [ ]:
from IPython.display import Video, display\nif os.path.isfile(OUTPUT) and os.path.getsize(OUTPUT) > 10000:\n    display(Video(OUTPUT, embed=True))\nelse:\n    print('No valid MP4 found. Log:', LOG)

## 5. Optional: send the run log to GitHub\n\nAdd a Colab Secret named `GH_TOKEN` containing a fine-grained token for this repository with **Contents: Read and write**. This uploads only the text log to the `colab-runs` branch. No model weights, credentials, or MP4 files are uploaded.

In [ ]:
try:\n    from google.colab import userdata\n    token = userdata.get('GH_TOKEN')\nexcept Exception:\n    token = None\nif not token:\n    print('GH_TOKEN not configured. Log remains at:', LOG)\nelse:\n    import requests, base64\n    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')\n    path = f'colab_runs/{stamp}_wan21_test.log'\n    with open(LOG, 'rb') as f: encoded = base64.b64encode(f.read()).decode('ascii')\n    api = f'https://api.github.com/repos/6d28nf7wz9-web/fitness-shorts-bot/contents/{path}'\n    response = requests.put(api, headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json'}, json={'message':f'Add Colab Wan2.1 run log {stamp}','content':encoded,'branch':'colab-runs'}, timeout=60)\n    print('GitHub log upload status:', response.status_code)\n    if response.ok: print('Saved log:', response.json().get('content',{}).get('html_url'))\n    else: print(response.text[:1000])\n    del token